# Aegis-v2 campaign on a Colab GPU

1. **Runtime -> Change runtime type -> T4 GPU**.
2. For a private repo: create a GitHub token (Settings -> Developer settings -> Fine-grained token, read-only *Contents* on the repo) and add it in Colab's **Secrets** panel (key icon, left) as `GH_TOKEN`, with notebook access enabled.
3. Set `GITHUB_REPO` below, then **Runtime -> Run all**. Results download as a zip at the end.

In [ ]:
GITHUB_REPO = "OWNER/aegis-v2"   # <- change to your repo
CAMPAIGN = "--days 3 --step-minutes 60 --seeds 0 1 2 --eval-per-class 15 --scrub-sweep-hours 1 3 6 24 --orbit tle"
NAME = "main_gpu"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os, subprocess
try:
    from google.colab import userdata
    token = userdata.get("GH_TOKEN")
except Exception:
    token = None  # public repo
url = f"https://{token}@github.com/{GITHUB_REPO}.git" if token else f"https://github.com/{GITHUB_REPO}.git"
if not os.path.isdir("aegis-v2"):
    subprocess.run(["git", "clone", "--depth", "1", url, "aegis-v2"], check=True)
%cd aegis-v2

In [ ]:
!pip install -q sgp4 ppigrf zstandard safetensors huggingface_hub

In [ ]:
!mkdir -p data/eurosat && cd data/eurosat && [ -d 2750 ] || (curl -sL -o EuroSAT.zip https://huggingface.co/datasets/torchgeo/eurosat/resolve/main/EuroSAT.zip && unzip -q EuroSAT.zip)
!ls data/eurosat/2750 | head -3

In [ ]:
!python -m pytest -q tests/test_correctness_fixes.py tests/test_orbit_model.py

In [ ]:
!python experiments/run_campaign.py $CAMPAIGN --device cuda --name $NAME 2>&1 | grep -v Warning

In [ ]:
!python experiments/make_figures.py --name $NAME
from IPython.display import Image, display
for f in ("accuracy_over_time", "pareto_bandwidth", "fault_timeline"):
    display(Image(f"results/{NAME}/figures/{f}.png"))

In [ ]:
import shutil
from google.colab import files
shutil.make_archive(f"results_{NAME}", "zip", "results", NAME)
files.download(f"results_{NAME}.zip")

Unzip the download into the local repo's `results/` folder, then run `streamlit run dashboard/app.py` locally to explore it.